# Task 5: Idiot Sandwich: TRL DPOTrainer with LoRA persona tuning

**Settings:** Accelerator **GPU T4 x2**, Internet **on**. No inputs needed.
Run with **Save Version > Save & Run All (Commit)**.

`HuggingFaceTB/SmolLM2-360M-Instruct` learns a Gordon Ramsay speaking style from the 220 provided
preference pairs (`data/persona_lora/ramsay_style_preferences.json`; 200 train, 20 held out). Chosen and
rejected answers give the same advice in different tones, so DPO learns the tone. Only rank-16 LoRA
adapters on every linear projection are trained (about 2.4% of the weights); TRL scores the reference by
switching the adapters off. Settings and their reasons: `configs/lora_persona.yaml`; code: `src/pa2/lora_persona.py`.

1. Train on GPU 0 with TRL's `DPOTrainer` (about 75 steps). The held-out pairs are evaluated before training
   (must give loss ln 2, because the adapter starts at zero) and every 5 steps.
2. Curves: loss, reward accuracy and held-out rewards.
3. Before and after: plain SmolLM2 (GPU 0) and SmolLM2 + adapter (GPU 1) answer 8 prompts from the persona
   evaluation set, with the decoding chosen in Task 3 (greedy, repetition penalty 1.1).

The adapter stays in this notebook's output (`outputs/lora/run/adapter`) for Task 6. Small files are archived as
`task5_lora_outputs.zip`.

In [ ]:
REPO_URL = "https://github.com/mabdullahi7780/small-model-alignment-starter.git"
BRANCH = "main"
WORK_DIR = "/kaggle/working"
SCRATCH_DIR = "/tmp"
REPO_DIR = f"{SCRATCH_DIR}/small-model-alignment-starter"
OUT_DIR = f"{WORK_DIR}/outputs/lora"
CONFIG_FILE = "configs/lora_persona.yaml"
BASE_MODEL = "HuggingFaceTB/SmolLM2-360M-Instruct"
EVAL_SET = "eval_sets/persona_style_eval_40.json"
SAMPLES_PER_CATEGORY = 2
RUN_SETTINGS = {"config": CONFIG_FILE, "base_model": BASE_MODEL, "samples_per_category": SAMPLES_PER_CATEGORY}

In [ ]:
import os
import subprocess
import sys


def run(cmd, cwd=None, env=None, check=True):
    print("$", " ".join(cmd), flush=True)
    return subprocess.run(cmd, cwd=cwd, env=env, check=check)


if not os.path.isdir(REPO_DIR):
    run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR])
run(["git", "log", "-1", "--format=commit %H%nauthor date %ad%nsubject %s"], cwd=REPO_DIR)

In [ ]:
# Kaggle's preinstalled torch is built for its GPU runtime; pin it so pip can never replace it.
TORCH_BEFORE = subprocess.run(
    [sys.executable, "-c", "import torch; print(torch.__version__)"],
    capture_output=True, text=True, check=True,
).stdout.strip()
print("Preinstalled torch:", TORCH_BEFORE)

constraint_file = f"{SCRATCH_DIR}/torch-constraint.txt"
with open(constraint_file, "w") as handle:
    handle.write(f"torch=={TORCH_BEFORE}\n")

run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "-c", constraint_file], cwd=REPO_DIR)
run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], cwd=REPO_DIR)

In [ ]:
import json

COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout.strip()
os.makedirs(f"{OUT_DIR}/logs", exist_ok=True)
with open(f"{OUT_DIR}/run_info.json", "w") as handle:
    json.dump({"commit": COMMIT, "torch": TORCH_BEFORE, **RUN_SETTINGS}, handle, indent=2)
print("commit", COMMIT)

In [ ]:
def run_on_gpus(jobs, tag):
    """Start one subprocess per job, each pinned to its own GPU, and wait for all of them.

    jobs: list of (name, gpu_index, argv). Each job's output goes to logs/<tag>_<name>.log.
    """
    # expandable_segments lets the CUDA allocator reuse freed memory instead of fragmenting it.
    env_base = {**os.environ, "PYTHONPATH": f"{REPO_DIR}/src:{REPO_DIR}", "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}
    running = []
    for name, gpu, argv in jobs:
        log_path = f"{OUT_DIR}/logs/{tag}_{name}.log"
        handle = open(log_path, "w")
        print(f"start {tag}/{name} on GPU {gpu}", flush=True)
        process = subprocess.Popen(
            [sys.executable, *argv], cwd=REPO_DIR, stdout=handle, stderr=subprocess.STDOUT,
            env={**env_base, "CUDA_VISIBLE_DEVICES": str(gpu)},
        )
        running.append((name, process, handle, log_path))
    failed = []
    for name, process, handle, log_path in running:
        code = process.wait()
        handle.close()
        print(f"\n===== {tag}/{name}: exit code {code} =====")
        print(open(log_path).read()[-8000:])
        if code:
            failed.append(name)
    if failed:
        raise RuntimeError(f"{tag}: failed jobs {failed}; see {OUT_DIR}/logs")

## 1. Train the LoRA adapter with TRL's DPOTrainer

In [ ]:
import math

run_on_gpus([("train", 0, ["-m", "pa2.lora_persona", "--config", CONFIG_FILE, "--out-dir", f"{OUT_DIR}/run"])], "lora")
summary = json.load(open(f"{OUT_DIR}/run/summary.json"))
start, end = summary["initial_eval"], summary["final_eval"]
print("log-prob kernel:", summary["logprob_kernel"])
print("trainable parameters:", summary["parameters"])
print(f"before training: loss {start['eval_loss']:.6f} (ln 2 = {math.log(2):.6f}), accuracy {start['eval_rewards/accuracies']}, "
      f"margin {start['eval_rewards/margins']:.2e}")
print(f"after training:  loss {end['eval_loss']:.4f}, accuracy {end['eval_rewards/accuracies']:.3f}, "
      f"margin {end['eval_rewards/margins']:.4f}, chosen {end['eval_rewards/chosen']:.4f}, rejected {end['eval_rewards/rejected']:.4f}")
if abs(start["eval_loss"] - math.log(2)) > 1e-3:
    raise RuntimeError("the adapted model must equal the base model before training (adapter B starts at zero)")
assert os.path.exists(f"{OUT_DIR}/run/adapter/adapter_model.safetensors")

## 2. Curves

In [ ]:
run([sys.executable, "-m", "pa2.plots", "lora", "--metrics", f"{OUT_DIR}/run/log_history.json", "--out", f"{OUT_DIR}/lora_curves.png"],
    cwd=REPO_DIR, env={**os.environ, "PYTHONPATH": f"{REPO_DIR}/src:{REPO_DIR}"})
from IPython.display import Image, display
display(Image(f"{OUT_DIR}/lora_curves.png"))

## 3. Before and after: plain SmolLM2 vs SmolLM2 + adapter

In [ ]:
eval_rows = json.load(open(f"{REPO_DIR}/{EVAL_SET}"))
samples = []
for category in ("cooking", "coding", "general", "safety"):
    samples += [row for row in eval_rows if row["category"] == category][:SAMPLES_PER_CATEGORY]
SAMPLE_FILE = f"{OUT_DIR}/sample_prompts.json"
json.dump(samples, open(SAMPLE_FILE, "w"), indent=2)

common = ["-m", "pa2.generate", "--model", BASE_MODEL, "--prompt-format", "chat_template", "--prompts", SAMPLE_FILE,
          "--batch-size", "8", "--max-new-tokens", "256", "--repetition-penalty", "1.1"]
run_on_gpus([
    ("base", 0, [*common, "--generator", "smollm2-360m-instruct", "--out", f"{OUT_DIR}/samples/base.json"]),
    ("lora", 1, [*common, "--generator", "smollm2-360m-instruct-lora-dpo", "--adapter", f"{OUT_DIR}/run/adapter",
                 "--out", f"{OUT_DIR}/samples/lora.json"]),
], "samples")

base = json.load(open(f"{OUT_DIR}/samples/base.json"))
lora = json.load(open(f"{OUT_DIR}/samples/lora.json"))
for before, after, row in zip(base, lora, samples):
    print("\n" + "=" * 100 + f"\n[{row['category']}] {row['instruction']}")
    print("\n--- plain SmolLM2:\n" + before["output"][:600])
    print("\n--- SmolLM2 + LoRA-DPO adapter:\n" + after["output"][:600])

## Archive the small outputs

In [ ]:
import zipfile


def zip_small_outputs(archive_path, root, skip_dirs=(), skip_suffixes=(".safetensors", ".jsonl.gz")):
    """Zip everything under root except checkpoints and bulky data, so the archive stays small."""
    count = 0
    with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as archive:
        for folder, dirs, files in os.walk(root):
            rel_folder = os.path.relpath(folder, root)
            dirs[:] = [d for d in dirs if os.path.normpath(os.path.join(rel_folder, d)) not in skip_dirs]
            for name in sorted(files):
                if name.endswith(skip_suffixes):
                    continue
                path = os.path.join(folder, name)
                archive.write(path, os.path.relpath(path, root))
                count += 1
    print(f"Saved {archive_path} ({count} files, {os.path.getsize(archive_path) / 1e6:.1f} MB)")


zip_small_outputs(f"{WORK_DIR}/task5_lora_outputs.zip", OUT_DIR, skip_dirs=("run/trainer",),
                  skip_suffixes=(".safetensors", ".bin"))